In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import holidays

In [12]:
# Cargar los datos de ventas y competencia desde data/raw/entrenamiento

# Cargar los archivos CSV en DataFrames
ventas_df = pd.read_csv("../data/raw/entrenamiento/ventas.csv")
competencia_df = pd.read_csv("../data/raw/entrenamiento/competencia.csv")

# Convertir las fechas a datetime
ventas_df["fecha"] = pd.to_datetime(ventas_df["fecha"])
competencia_df["fecha"] = pd.to_datetime(competencia_df["fecha"])

# Mostrar las primeras filas para verificar
print("Ventas:")
print(ventas_df.head())
print("\nCompetencia:")
print(competencia_df.head())

Ventas:
       fecha producto_id                            nombre categoria  \
0 2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1 2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2 2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3 2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4 2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True                  6   
1  Zapatillas Running          135         True                 10   
2  Zapatillas Running           85        False                  2   
3  Zapatillas Running           75        False                  2   
4        Ropa Running           35        False                  2   

   precio_venta  ingresos  
0        118.36    710.16  
1        136.82   1368.20  
2         84.93    169.86  
3         75.42    150.84 

In [13]:
resumen_calidad = pd.DataFrame({
    "tipo": ventas_df.dtypes.astype(str),
    "no_nulos": ventas_df.notna().sum(),
    "nulos": ventas_df.isna().sum(),
    "porcentaje_nulos": (ventas_df.isna().mean() * 100).round(2),
    "valores_unicos": ventas_df.nunique(dropna=True),
})

filas_duplicadas = ventas_df.duplicated().sum()
duplicados_fecha_producto = ventas_df.duplicated(
    subset=["fecha", "producto_id"]
).sum()
fechas_parseadas = pd.to_datetime(ventas_df["fecha"], errors="coerce")
fechas_invalidas = fechas_parseadas.isna().sum()
columnas_texto = [
    columna
    for columna in ventas_df.columns
    if pd.api.types.is_string_dtype(ventas_df[columna].dtype)
]
cadenas_vacias = ventas_df[columnas_texto].apply(
    lambda columna: columna.str.strip().eq("").sum()
)
columnas_numericas = ventas_df.select_dtypes(include="number").columns
valores_negativos = ventas_df[columnas_numericas].lt(0).sum()
inconsistencias_ingresos = (
    ventas_df["ingresos"]
    - ventas_df["precio_venta"] * ventas_df["unidades_vendidas"]
).abs().gt(0.01).sum()
descriptivos = ventas_df.describe(include="all").T

print(f"Dimensiones: {ventas_df.shape[0]} filas x {ventas_df.shape[1]} columnas")
print(f"Rango de fechas: {fechas_parseadas.min().date()} a {fechas_parseadas.max().date()}")
print(f"Filas duplicadas completas: {filas_duplicadas}")
print(f"Claves fecha-producto duplicadas: {duplicados_fecha_producto}")
print(f"Fechas no interpretables: {fechas_invalidas}")
print(f"Inconsistencias ingresos vs. precio_venta x unidades: {inconsistencias_ingresos}")
print("\nCadenas vacías por columna de texto:")
print(cadenas_vacias.to_string())
print("\nValores negativos por columna numérica:")
print(valores_negativos.to_string())
print("\nCalidad por variable:")
print(resumen_calidad.to_string())
print("\nEstadísticos descriptivos:")
print(descriptivos.to_string())

Dimensiones: 3552 filas x 10 columnas
Rango de fechas: 2021-10-25 a 2024-11-30
Filas duplicadas completas: 0
Claves fecha-producto duplicadas: 0
Fechas no interpretables: 0
Inconsistencias ingresos vs. precio_venta x unidades: 0

Cadenas vacías por columna de texto:
producto_id     0
nombre          0
categoria       0
subcategoria    0

Valores negativos por columna numérica:
precio_base          0
unidades_vendidas    0
precio_venta         0
ingresos             0

Calidad por variable:
                             tipo  no_nulos  nulos  porcentaje_nulos  valores_unicos
fecha              datetime64[us]      3552      0               0.0             148
producto_id                   str      3552      0               0.0              24
nombre                        str      3552      0               0.0              24
categoria                     str      3552      0               0.0               4
subcategoria                  str      3552      0               0.0            

In [11]:
resumen_calidad_competencia = pd.DataFrame({
    "tipo": competencia_df.dtypes.astype(str),
    "no_nulos": competencia_df.notna().sum(),
    "nulos": competencia_df.isna().sum(),
    "porcentaje_nulos": (competencia_df.isna().mean() * 100).round(2),
    "valores_unicos": competencia_df.nunique(dropna=True),
})

filas_duplicadas_competencia = competencia_df.duplicated().sum()
duplicados_fecha_producto_competencia = competencia_df.duplicated(
    subset=["fecha", "producto_id"]
).sum()
fechas_competencia_parseadas = pd.to_datetime(
    competencia_df["fecha"], errors="coerce"
)
fechas_invalidas_competencia = fechas_competencia_parseadas.isna().sum()
columnas_texto_competencia = [
    columna
    for columna in competencia_df.columns
    if pd.api.types.is_string_dtype(competencia_df[columna].dtype)
]
cadenas_vacias_competencia = competencia_df[columnas_texto_competencia].apply(
    lambda columna: columna.str.strip().eq("").sum()
)
columnas_numericas_competencia = competencia_df.select_dtypes(
    include="number"
).columns
valores_negativos_competencia = competencia_df[
    columnas_numericas_competencia
].lt(0).sum()
descriptivos_competencia = competencia_df.describe(include="all").T

print(
    f"Dimensiones: {competencia_df.shape[0]} filas x "
    f"{competencia_df.shape[1]} columnas"
)
print(
    "Rango de fechas: "
    f"{fechas_competencia_parseadas.min().date()} a "
    f"{fechas_competencia_parseadas.max().date()}"
)
print(f"Filas duplicadas completas: {filas_duplicadas_competencia}")
print(
    "Claves fecha-producto duplicadas: "
    f"{duplicados_fecha_producto_competencia}"
)
print(f"Fechas no interpretables: {fechas_invalidas_competencia}")
print("\nCadenas vacías por columna de texto:")
print(cadenas_vacias_competencia.to_string())
print("\nValores negativos por columna numérica:")
print(valores_negativos_competencia.to_string())
print("\nCalidad por variable:")
print(resumen_calidad_competencia.to_string())
print("\nEstadísticos descriptivos:")
print(descriptivos_competencia.to_string())

Dimensiones: 3552 filas x 5 columnas
Rango de fechas: 2021-10-25 a 2024-11-30
Filas duplicadas completas: 0
Claves fecha-producto duplicadas: 0
Fechas no interpretables: 0

Cadenas vacías por columna de texto:
producto_id    0

Valores negativos por columna numérica:
Amazon          0
Decathlon       0
Deporvillage    0

Calidad por variable:
                        tipo  no_nulos  nulos  porcentaje_nulos  valores_unicos
fecha         datetime64[us]      3552      0               0.0             148
producto_id              str      3552      0               0.0              24
Amazon               float64      3552      0               0.0            3125
Decathlon            float64      3552      0               0.0            3001
Deporvillage         float64      3552      0               0.0            3105

Estadísticos descriptivos:
               count unique       top freq                 mean                  min                  25%                  50%                  75%

In [14]:
ventas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3552 entries, 0 to 3551
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fecha              3552 non-null   datetime64[us]
 1   producto_id        3552 non-null   str           
 2   nombre             3552 non-null   str           
 3   categoria          3552 non-null   str           
 4   subcategoria       3552 non-null   str           
 5   precio_base        3552 non-null   int64         
 6   es_estrella        3552 non-null   bool          
 7   unidades_vendidas  3552 non-null   int64         
 8   precio_venta       3552 non-null   float64       
 9   ingresos           3552 non-null   float64       
dtypes: bool(1), datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 433.1 KB


In [15]:
competencia_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3552 entries, 0 to 3551
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   fecha         3552 non-null   datetime64[us]
 1   producto_id   3552 non-null   str           
 2   Amazon        3552 non-null   float64       
 3   Decathlon     3552 non-null   float64       
 4   Deporvillage  3552 non-null   float64       
dtypes: datetime64[us](1), float64(3), str(1)
memory usage: 166.6 KB


In [17]:
df = ventas_df.merge(
    competencia_df,
    on=["fecha", "producto_id"],
    how="left",
    validate="one_to_one",
)

print(f"Dimensiones de df: {df.shape[0]} filas x {df.shape[1]} columnas")
print("Nulos en los precios de competencia después de la unión:")
print(df[["Amazon", "Decathlon", "Deporvillage"]].isna().sum().to_string())
df.head()

Dimensiones de df: 3552 filas x 13 columnas
Nulos en los precios de competencia después de la unión:
Amazon          0
Decathlon       0
Deporvillage    0


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2021-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,6,118.36,710.16,82.96,111.88,97.43
1,2021-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,10,136.82,1368.20,112.56,108.61,115.58
2,2021-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,2,84.93,169.86,79.79,78.44,80.11
3,2021-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,2,75.42,150.84,72.60,67.29,74.45
4,2021-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,2,35.87,71.74,37.71,33.60,33.07
